In [1]:

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("GTFS_Timetable_Preprocessing")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# reducing unnecessary Spark log messages.
spark.sparkContext.setLogLevel("WARN")
print("PySpark version:", spark.version)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/12 20:17:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark version: 3.5.1
Shuffle partitions: 4
Spark UI: http://192.168.1.13:4040


In [2]:

from pathlib import Path
#  folder where the notebook is currently running.
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
gtfs_folder = (
    project_root
    / "data"
    / "raw"
    / "timetable"
    / "extracted"
)

#  where the processed timetable will be saved.
processed_folder = (
    project_root
    / "data"
    / "processed"
    / "timetable"
)

print("Current folder:", current_folder)
print("Project root:", project_root)
print("GTFS folder:", gtfs_folder)
print("GTFS folder exists:", gtfs_folder.exists())

Current folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/notebooks
Project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
GTFS folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/timetable/extracted
GTFS folder exists: True


In [3]:
# main GTFS files required for timetable analysis.
required_files = [
    "agency.txt",
    "routes.txt",
    "trips.txt",
    "stop_times.txt",
    "stops.txt"
]
missing_files = []
for filename in required_files:
    file_path = gtfs_folder / filename

    if file_path.exists():
        print("Found:", filename)
    else:
        missing_files.append(filename)

# stoop notebook when an important file is missing.
if missing_files:
    raise FileNotFoundError(
        f"Missing GTFS files: {missing_files}"
    )

print("All required GTFS files are available.")

Found: agency.txt
Found: routes.txt
Found: trips.txt
Found: stop_times.txt
Found: stops.txt
All required GTFS files are available.


## GTFS loading function

In [4]:
def load_gtfs_file(filename):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .csv(str(gtfs_folder / filename))
    )
print("GTFS loader created successfully.")

GTFS loader created successfully.


## Load timetable tables

In [5]:
#  operator information
agency_df = load_gtfs_file("agency.txt")
#  route information
routes_df = load_gtfs_file("routes.txt")
#  scheduled journey information.
trips_df = load_gtfs_file("trips.txt")
#  arrival and departure time for every trip and stop
stop_times_df = load_gtfs_file("stop_times.txt")
#  stop names and geographical coordinates
stops_df = load_gtfs_file("stops.txt")
print("All GTFS tables loaded successfully.")

[Stage 4:>                                                          (0 + 1) / 1]

All GTFS tables loaded successfully.


In [6]:
# storing all DataFrames in a dictionary so they can be checked together
datasets = {
    "agency": agency_df,
    "routes": routes_df,
    "trips": trips_df,
    "stop_times": stop_times_df,
    "stops": stops_df
}
dataset_counts = {}
#  rows and inspect the initial number of partitions
for name, dataframe in datasets.items():
    row_count = dataframe.count()
    partition_count = dataframe.rdd.getNumPartitions()
    dataset_counts[name] = row_count
    print(
        f"{name}: {row_count:,} rows "
        f"| {partition_count} partitions"
    )

agency: 60 rows | 1 partitions


routes: 1,171 rows | 1 partitions


trips: 135,003 rows | 5 partitions


stop_times: 5,370,199 rows | 8 partitions


[Stage 17:>                                                         (0 + 1) / 1]

stops: 30,519 rows | 1 partitions


In [7]:
# showing the column names in every GTFS table.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()} COLUMNS")
    print(dataframe.columns)


AGENCY COLUMNS
['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'agency_noc']

ROUTES COLUMNS
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']

TRIPS COLUMNS
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'vehicle_journey_code']

STOP_TIMES COLUMNS
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'timepoint']

STOPS COLUMNS
['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon', 'wheelchair_boarding', 'location_type', 'parent_station', 'platform_code']


In [8]:
print("AGENCY SAMPLE")
agency_df.show(5, truncate=False)
print("ROUTES SAMPLE")
routes_df.show(5, truncate=False)
print("TRIPS SAMPLE")
trips_df.show(5, truncate=False)
print("STOP TIMES SAMPLE")
stop_times_df.show(5, truncate=False)
print("STOPS SAMPLE")
stops_df.show(5, truncate=False)

AGENCY SAMPLE


+---------+-------------------------+--------------------------+---------------+-----------+------------+----------+
|agency_id|agency_name              |agency_url                |agency_timezone|agency_lang|agency_phone|agency_noc|
+---------+-------------------------+--------------------------+---------------+-----------+------------+----------+
|OP101    |Select Bus Services      |https://www.traveline.info|Europe/London  |EN         |NULL        |SLBS      |
|OP102    |LandFlight               |https://www.traveline.info|Europe/London  |EN         |NULL        |SLVL      |
|OP10232  |Solus Coaches            |https://www.traveline.info|Europe/London  |EN         |NULL        |SOLU      |
|OP103    |National Express Coventry|https://www.traveline.info|Europe/London  |EN         |NULL        |TCVW      |
|OP104    |Let's Go                 |https://www.traveline.info|Europe/London  |EN         |NULL        |TEXP      |
+---------+-------------------------+--------------------------+

+--------+---------+----------------+---------------+----------+
|route_id|agency_id|route_short_name|route_long_name|route_type|
+--------+---------+----------------+---------------+----------+
|211     |OP850    |740             |NULL           |3         |
|541     |OP87     |X15             |NULL           |3         |
|542     |OP87     |X75             |NULL           |3         |
|555     |OP90     |226             |NULL           |3         |
|560     |OP90     |26A             |NULL           |3         |
+--------+---------+----------------+---------------+----------+
only showing top 5 rows

TRIPS SAMPLE
+--------+----------+------------------------------------------+--------------+------------+--------+--------+---------------------+--------------------+
|route_id|service_id|trip_id                                   |trip_headsign |direction_id|block_id|shape_id|wheelchair_accessible|vehicle_journey_code|
+--------+----------+------------------------------------------+-----

In [9]:
# schema of every loaded DataFrame.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()} SCHEMA")
    dataframe.printSchema()


AGENCY SCHEMA
root
 |-- agency_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_url: string (nullable = true)
 |-- agency_timezone: string (nullable = true)
 |-- agency_lang: string (nullable = true)
 |-- agency_phone: string (nullable = true)
 |-- agency_noc: string (nullable = true)


ROUTES SCHEMA
root
 |-- route_id: string (nullable = true)
 |-- agency_id: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- route_type: string (nullable = true)


TRIPS SCHEMA
root
 |-- route_id: string (nullable = true)
 |-- service_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- trip_headsign: string (nullable = true)
 |-- direction_id: string (nullable = true)
 |-- block_id: string (nullable = true)
 |-- shape_id: string (nullable = true)
 |-- wheelchair_accessible: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)


STOP_TIMES SCHEMA
root

## checking missing values

In [10]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    sum as spark_sum
)
def show_missing_values(dataframe, columns, dataset_name):
    """
    Count null and blank values in selected columns.
    """
    #  examine columns that actually exist
    available_columns = [
        column_name
        for column_name in columns
        if column_name in dataframe.columns
    ]
    expressions = []
    for column_name in available_columns:
        missing_condition = (
            col(column_name).isNull()
            | (trim(col(column_name)) == "")
        )
        expressions.append(
            spark_sum(
                when(missing_condition, 1).otherwise(0)
            ).alias(column_name)
        )
    print(f"Missing values in {dataset_name}:")
    dataframe.select(expressions).show(
        truncate=False
    )
show_missing_values(
    agency_df,
    ["agency_id", "agency_name", "agency_noc"],
    "agency"
)
show_missing_values(
    routes_df,
    [
        "route_id",
        "agency_id",
        "route_short_name",
        "route_long_name"
    ],
    "routes"
)
show_missing_values(
    trips_df,
    [
        "trip_id",
        "route_id",
        "service_id",
        "vehicle_journey_code"
    ],
    "trips"
)
show_missing_values(
    stop_times_df,
    [
        "trip_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "stop_sequence"
    ],
    "stop_times"
)
show_missing_values(
    stops_df,
    [
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ],
    "stops"
)

Missing values in agency:
+---------+-----------+----------+
|agency_id|agency_name|agency_noc|
+---------+-----------+----------+
|0        |0          |0         |
+---------+-----------+----------+

Missing values in routes:
+--------+---------+----------------+---------------+
|route_id|agency_id|route_short_name|route_long_name|
+--------+---------+----------------+---------------+
|0       |0        |0               |1171           |
+--------+---------+----------------+---------------+

Missing values in trips:


+-------+--------+----------+--------------------+
|trip_id|route_id|service_id|vehicle_journey_code|
+-------+--------+----------+--------------------+
|0      |0       |0         |0                   |
+-------+--------+----------+--------------------+

Missing values in stop_times:


+-------+-------+------------+--------------+-------------+
|trip_id|stop_id|arrival_time|departure_time|stop_sequence|
+-------+-------+------------+--------------+-------------+
|0      |0      |0           |0             |0            |
+-------+-------+------------+--------------+-------------+

Missing values in stops:
+-------+---------+--------+--------+
|stop_id|stop_name|stop_lat|stop_lon|
+-------+---------+--------+--------+
|0      |0        |0       |0       |
+-------+---------+--------+--------+



## checking duplicates

In [11]:
def duplicate_key_count(dataframe, key_columns):
    total_count = dataframe.count()
    unique_count = (
        dataframe
        .select(*key_columns)
        .dropDuplicates()
        .count()
    )
    return total_count - unique_count
print(
    "Duplicate agency IDs:",
    duplicate_key_count(
        agency_df,
        ["agency_id"]
    )
)
print(
    "Duplicate route IDs:",
    duplicate_key_count(
        routes_df,
        ["route_id"]
    )
)
print(
    "Duplicate trip IDs:",
    duplicate_key_count(
        trips_df,
        ["trip_id"]
    )
)
print(
    "Duplicate stop IDs:",
    duplicate_key_count(
        stops_df,
        ["stop_id"]
    )
)
print(
    "Duplicate trip-stop sequences:",
    duplicate_key_count(
        stop_times_df,
        ["trip_id", "stop_sequence"]
    )
)

Duplicate agency IDs: 0
Duplicate route IDs: 0


Duplicate trip IDs: 0
Duplicate stop IDs: 0


26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/12 20:26:59 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate trip-stop sequences: 0


In [12]:
def trim_string_columns(dataframe):
    cleaned_df = dataframe
    for column_name, data_type in dataframe.dtypes:
        if data_type == "string":
            cleaned_df = cleaned_df.withColumn(
                column_name,
                trim(col(column_name))
            )
    return cleaned_df
# clean leading and trailing spaces
agency_clean = trim_string_columns(agency_df)
routes_clean = trim_string_columns(routes_df)
trips_clean = trim_string_columns(trips_df)
stop_times_clean = trim_string_columns(stop_times_df)
stops_clean = trim_string_columns(stops_df)
print("String columns cleaned successfully.")

String columns cleaned successfully.


In [13]:
# keep one record for each operator
agency_clean = agency_clean.dropDuplicates(
    ["agency_id"]
)
# keep one record for each route
routes_clean = routes_clean.dropDuplicates(
    ["route_id"]
)
# keep one record for each trip
trips_clean = trips_clean.dropDuplicates(
    ["trip_id"]
)
# keep one record for each stop
stops_clean = stops_clean.dropDuplicates(
    ["stop_id"]
)
#  trip can visit many stops, so the combined key is used
stop_times_clean = stop_times_clean.dropDuplicates(
    ["trip_id", "stop_sequence"]
)
print("Duplicate key records removed.")

Duplicate key records removed.


## convert numeric columns

In [14]:
# convert route type when the column exists
if "route_type" in routes_clean.columns:
    routes_clean = routes_clean.withColumn(
        "route_type",
        col("route_type").cast("int")
    )
# change stop sequence into an integer
stop_times_clean = stop_times_clean.withColumn(
    "stop_sequence",
    col("stop_sequence").cast("int")
)
# convert stop coordinates into numeric values
stops_clean = (
    stops_clean
    .withColumn(
        "stop_lat",
        col("stop_lat").cast("double")
    )
    .withColumn(
        "stop_lon",
        col("stop_lon").cast("double")
    )
)
print("Important numeric columns converted.")

Important numeric columns converted.


## GTFS times into seconds

In [15]:
from pyspark.sql.functions import split
# splitting arrival time into hours, minutes and seconds
arrival_parts = split(
    col("arrival_time"),
    ":"
)
#  departure time into hours, minutes and seconds
departure_parts = split(
    col("departure_time"),
    ":"
)
stop_times_clean = (
    stop_times_clean
    .withColumn(
        "arrival_seconds",
        arrival_parts.getItem(0).cast("int") * 3600
        + arrival_parts.getItem(1).cast("int") * 60
        + arrival_parts.getItem(2).cast("int")
    )
    .withColumn(
        "departure_seconds",
        departure_parts.getItem(0).cast("int") * 3600
        + departure_parts.getItem(1).cast("int") * 60
        + departure_parts.getItem(2).cast("int")
    )
)
stop_times_clean.select(
    "trip_id",
    "stop_id",
    "arrival_time",
    "arrival_seconds",
    "departure_time",
    "departure_seconds"
).show(5, truncate=False)

[Stage 87:>                                                         (0 + 1) / 1]

+------------------------------------------+-----------+------------+---------------+--------------+-----------------+
|trip_id                                   |stop_id    |arrival_time|arrival_seconds|departure_time|departure_seconds|
+------------------------------------------+-----------+------------+---------------+--------------+-----------------+
|VJ00005702ab013a85402a8f191499c654a7642e7d|4200F165421|07:33:00    |27180          |07:33:00      |27180            |
|VJ00005702ab013a85402a8f191499c654a7642e7d|4200F207301|07:41:00    |27660          |07:41:00      |27660            |
|VJ00005702ab013a85402a8f191499c654a7642e7d|4200F226101|07:55:00    |28500          |07:55:00      |28500            |
|VJ00005702ab013a85402a8f191499c654a7642e7d|4200F127012|07:36:00    |27360          |07:36:00      |27360            |
|VJ00005702ab013a85402a8f191499c654a7642e7d|4200F126981|07:38:00    |27480          |07:38:00      |27480            |
+------------------------------------------+----

## Join trips and routes

In [16]:
from pyspark.sql.functions import broadcast
# routes is much smaller than trips, so Spark can broadcast it
trip_route_df = (
    trips_clean
    .join(
        broadcast(routes_clean),
        on="route_id",
        how="inner"
    )
)
print(
    "Trips joined with routes:",
    f"{trip_route_df.count():,}"
)

[Stage 93:>                                                         (0 + 4) / 4]

Trips joined with routes: 135,003


## Join operator information

In [17]:
# adding operator information when agency_id exists
if (
    "agency_id" in trip_route_df.columns
    and "agency_id" in agency_clean.columns
):
    trip_details_df = (
        trip_route_df
        .join(
            broadcast(agency_clean),
            on="agency_id",
            how="left"
        )
    )
else:
    # continuing without agency join when agency_id is unavailable
    trip_details_df = trip_route_df

print(
    "Trip detail rows:",
    f"{trip_details_df.count():,}"
)

Trip detail rows: 135,003


## Join stop times

In [18]:
# join every scheduled stop time with its trip and route details
timetable_df = (
    stop_times_clean
    .join(
        trip_details_df,
        on="trip_id",
        how="inner"
    )
)
print("Stop times joined with trip details.")

Stop times joined with trip details.


## Join stop names and coordinates

In [19]:
# stops is small enough to be broadcast to the larger timetable table
timetable_df = (
    timetable_df
    .join(
        broadcast(stops_clean),
        on="stop_id",
        how="left"
    )
)
print("Stops joined successfully.")

Stops joined successfully.


In [20]:
#  most useful columns for later integration
# with live vehicle-location data
preferred_columns = [
    "agency_id",
    "agency_name",
    "agency_noc",
    "route_id",
    "route_short_name",
    "route_long_name",
    "service_id",
    "trip_id",
    "vehicle_journey_code",
    "trip_headsign",
    "direction_id",
    "stop_id",
    "stop_code",
    "stop_name",
    "stop_sequence",
    "arrival_time",
    "departure_time",
    "arrival_seconds",
    "departure_seconds",
    "stop_lat",
    "stop_lon",
    "shape_id"
]
#  only columns that are actually available in this dataset
available_columns = [
    column_name
    for column_name in preferred_columns
    if column_name in timetable_df.columns
]
timetable_df = timetable_df.select(
    *available_columns
)
print("Selected timetable columns:")
print(timetable_df.columns)

Selected timetable columns:
['agency_id', 'agency_name', 'agency_noc', 'route_id', 'route_short_name', 'route_long_name', 'service_id', 'trip_id', 'vehicle_journey_code', 'trip_headsign', 'direction_id', 'stop_id', 'stop_code', 'stop_name', 'stop_sequence', 'arrival_time', 'departure_time', 'arrival_seconds', 'departure_seconds', 'stop_lat', 'stop_lon', 'shape_id']


In [21]:
# redistribute the joined records across four Spark partitions
timetable_df = timetable_df.repartition(
    4,
    "route_id"
)
print(
    "Partitions after repartition:",
    timetable_df.rdd.getNumPartitions()
)

[Stage 122:>                                                        (0 + 4) / 4]

Partitions after repartition: 4


## Cache and count

In [24]:
# cache the joined timetable because it will be reused
timetable_df.cache()
# count() triggers Spark's lazy execution and places data in cache
joined_count = timetable_df.count()
print(
    "Joined timetable rows:",
    f"{joined_count:,}"
)
print(
    "Cached partitions:",
    timetable_df.rdd.getNumPartitions()
)

26/07/12 20:35:25 WARN CacheManager: Asked to cache already cached data.


Joined timetable rows: 5,370,199
Cached partitions: 4


In [25]:
timetable_df.show(
    10,
    truncate=False
)

+---------+-------------------------+----------+--------+----------------+---------------+----------+------------------------------------------+--------------------+-------------+------------+-----------+---------+----------------------+-------------+------------+--------------+---------------+-----------------+------------+----------+------------------------------------+
|agency_id|agency_name              |agency_noc|route_id|route_short_name|route_long_name|service_id|trip_id                                   |vehicle_journey_code|trip_headsign|direction_id|stop_id    |stop_code|stop_name             |stop_sequence|arrival_time|departure_time|arrival_seconds|departure_seconds|stop_lat    |stop_lon  |shape_id                            |
+---------+-------------------------+----------+--------+----------------+---------------+----------+------------------------------------------+--------------------+-------------+------------+-----------+---------+----------------------+-------------

In [26]:
print(
    "Distinct routes:",
    timetable_df.select(
        "route_id"
    ).distinct().count()
)
print(
    "Distinct trips:",
    timetable_df.select(
        "trip_id"
    ).distinct().count()
)
print(
    "Distinct stops:",
    timetable_df.select(
        "stop_id"
    ).distinct().count()
)
if "agency_id" in timetable_df.columns:
    print(
        "Distinct operators:",
        timetable_df.select(
            "agency_id"
        ).distinct().count()
    )

Distinct routes: 1171


Distinct trips: 135003
Distinct stops: 30417
Distinct operators: 60


## PySpark SQL query for joining

In [27]:
# registering the DataFrame as a temporary SQL table
timetable_df.createOrReplaceTempView(
    "timetable"
)
# SQL Query 1:
# count scheduled stop records and scheduled trips for each route
route_schedule_summary = spark.sql("""
    SELECT
        route_id,
        route_short_name,
        COUNT(*) AS scheduled_stop_records,
        COUNT(DISTINCT trip_id) AS scheduled_trips
    FROM timetable
    GROUP BY route_id, route_short_name
    ORDER BY scheduled_stop_records DESC
    LIMIT 10
""")
route_schedule_summary.show(
    truncate=False
)

26/07/12 20:41:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------+----------------+----------------------+---------------+
|route_id|route_short_name|scheduled_stop_records|scheduled_trips|
+--------+----------------+----------------------+---------------+
|24211   |11C             |93127                 |1195           |
|128663  |11A             |90526                 |1205           |
|6860    |74              |84116                 |2538           |
|13213   |50              |77550                 |2068           |
|4440    |14              |59886                 |1288           |
|128640  |6               |57673                 |1303           |
|627     |97              |56013                 |1137           |
|128731  |1               |55495                 |1064           |
|7399    |79              |53928                 |1157           |
|93860   |148             |52436                 |720            |
+--------+----------------+----------------------+---------------+



In [28]:
# showing how Spark plans to execute the timetable operations
timetable_df.explain(
    mode="formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (121)
+- == Final Plan ==
   ShuffleQueryStage (69), Statistics(sizeInBytes=2.1 GiB, rowCount=5.37E+6)
   +- Exchange (68)
      +- * Project (67)
         +- * BroadcastHashJoin LeftOuter BuildRight (66)
            :- * Project (53)
            :  +- * SortMergeJoin Inner (52)
            :     :- * Sort (13)
            :     :  +- ShuffleQueryStage (12), Statistics(sizeInBytes=737.5 MiB, rowCount=5.37E+6)
            :     :     +- Exchange (11)
            :     :        +- * Project (10)
            :     :           +- SortAggregate (9)
            :     :              +- * Sort (8)
            :     :                 +- ShuffleQueryStage (7), Statistics(sizeInBytes=819.4 MiB, rowCount=5.37E+6)
            :     :                    +- Exchange (6)
            :     :                       +- SortAggregate (5)
            :     :                          +- * Sort (4)
            :     :                             +- * Project (3)
         

In [29]:
#   processed timetable folder if it does not exist
processed_folder.mkdir(
    parents=True,
    exist_ok=True
)
# saving cleaned and joined timetable DataFrame
# overwrite mode replaces the previous output when the cell is rerun
(
    timetable_df.write
    .mode("overwrite")
    .parquet(str(processed_folder))
)
print("Processed timetable saved successfully.")
print("Saved location:", processed_folder)

[Stage 257:==========================================>              (3 + 1) / 4]

Processed timetable saved successfully.
Saved location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/timetable


In [30]:
# read the saved Parquet files back into PySpark
saved_timetable_df = spark.read.parquet(
    str(processed_folder)
)
# count the saved records
saved_count = saved_timetable_df.count()
print(
    "Original joined rows:",
    f"{joined_count:,}"
)
print(
    "Saved Parquet rows:",
    f"{saved_count:,}"
)
print(
    "Saved Parquet partitions:",
    saved_timetable_df.rdd.getNumPartitions()
)
# check whether any records were lost while saving
if saved_count != joined_count:
    raise ValueError(
        "Saved row count does not match the original joined row count."
    )
print("Parquet verification successful.")

Original joined rows: 5,370,199
Saved Parquet rows: 5,370,199
Saved Parquet partitions: 8
Parquet verification successful.


In [31]:
# display the files created inside the processed timetable folder
parquet_files = sorted(
    processed_folder.glob("*")
)
print("Files created:", len(parquet_files))
for file_path in parquet_files:
    print(file_path.name)

Files created: 10
._SUCCESS.crc
.part-00000-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet.crc
.part-00001-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet.crc
.part-00002-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet.crc
.part-00003-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet.crc
_SUCCESS
part-00000-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet
part-00001-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet
part-00002-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet
part-00003-1de6108d-0ffe-47f8-8281-c91c2d995dc5-c000.snappy.parquet


In [32]:
#  final summary of the processed timetable
print("TIMETABLE PREPROCESSING SUMMARY")
print("-" * 40)
print("Total rows:", f"{joined_count:,}")
print(
    "Distinct routes:",
    timetable_df.select("route_id").distinct().count()
)
print(
    "Distinct trips:",
    timetable_df.select("trip_id").distinct().count()
)
print(
    "Distinct stops:",
    timetable_df.select("stop_id").distinct().count()
)
print(
    "Partitions:",
    timetable_df.rdd.getNumPartitions()
)
print("Saved format: Parquet")
print("Saved location:", processed_folder)

TIMETABLE PREPROCESSING SUMMARY
----------------------------------------
Total rows: 5,370,199
Distinct routes: 1171
Distinct trips: 135003
Distinct stops: 30417
Partitions: 4
Saved format: Parquet
Saved location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/timetable
